# Franka 运动学与坐标约定

比较 MDH、DH、PoE 的末端位姿，以及几何/解析雅可比，并用 MuJoCo 核对。范围到 `link7`，不含夹爪。按顺序运行，不做动力学采集，也不写文件。

部分测试姿态超出实际关节范围，只用于运动学代数检查。

![Franka MDH 坐标系](dh-diagram-frankarobotics.png)

In [ ]:
from pathlib import Path
import sys

import numpy as np
import sympy
import mujoco

# 根据项目标记定位根目录，支持从仓库或 notebook 目录启动。
def _project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "robot_model" / "__init__.py"
        ).is_file():
            return candidate
    raise FileNotFoundError("找不到含 pyproject.toml 和 src/robot_model 的项目根目录")


_ROOT = _project_root(Path.cwd().resolve())
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from robot_model import Robot
from robot_model.robots.franka import PANDA_MDH_PARMS, PANDA_MDH_BODY_NAMES, panda_xml_path

panda_mdh = Robot.from_mdh("franka_panda_arm", PANDA_MDH_PARMS)
panda_dh = panda_mdh.to_dh()
panda_poe = panda_mdh.to_poe()
mdh_from_poe = panda_poe.to_mdh()
dh_from_poe = panda_poe.to_dh()

## 1. 三种模型描述

In [2]:
print("=== 原始 MDH ===")
panda_mdh.print_dh_table()
print("\n=== Craig 平移 → DH ===")
panda_dh.print_dh_table()
print("\n=== MDH → PoE（空间旋量）===")
panda_poe.print_poe_table()
print("\n=== PoE → MDH（反解校验）===")
mdh_from_poe.print_dh_table()
print("\n=== PoE → DH（反解 + Craig）===")
dh_from_poe.print_dh_table()

=== 原始 MDH ===


Joint,a,d,α,θ,关节类型
Joint1,0,0.333,0,q1,转动关节
Joint2,0,0,-pi/2,q2,转动关节
Joint3,0,0.316,pi/2,q3,转动关节
Joint4,0.0825,0,pi/2,q4,转动关节
Joint5,-0.0825,0.384,-pi/2,q5,转动关节
Joint6,0,0,pi/2,q6,转动关节
Joint7,0.088,0,pi/2,q7,转动关节



=== Craig 平移 → DH ===


Joint,a,d,α,θ,关节类型
Joint1,0,0.333,-pi/2,q1,转动关节
Joint2,0,0,pi/2,q2,转动关节
Joint3,0.0825,0.316,pi/2,q3,转动关节
Joint4,-0.0825,0,-pi/2,q4,转动关节
Joint5,0,0.384,pi/2,q5,转动关节
Joint6,0.088,0,pi/2,q6,转动关节
Joint7,0,0,0,q7,转动关节



=== MDH → PoE（空间旋量）===


Joint,ωx,ωy,ωz,vx,vy,vz,关节类型
Joint1,0,0,1,0,0,0,转动关节
Joint2,0,1,0,-0.333,0,0,转动关节
Joint3,0,0,1,0,0,0,转动关节
Joint4,0,-1,0,0.649,0,-0.0825,转动关节
Joint5,0,0,1,0,0,0,转动关节
Joint6,0,-1,0,1.033,0,0,转动关节
Joint7,0,0,-1,0,0.088,0,转动关节



=== PoE → MDH（反解校验）===


Joint,a,d,α,θ,关节类型
Joint1,0,0.333,0,q1,转动关节
Joint2,0,0,-pi/2,q2,转动关节
Joint3,0,0.316,pi/2,q3,转动关节
Joint4,0.0825,0,pi/2,q4,转动关节
Joint5,-0.0825,0.384,-pi/2,q5,转动关节
Joint6,0,0,pi/2,q6,转动关节
Joint7,0.088,0,pi/2,q7,转动关节



=== PoE → DH（反解 + Craig）===


Joint,a,d,α,θ,关节类型
Joint1,0,0.333,-pi/2,q1,转动关节
Joint2,0,0,pi/2,q2,转动关节
Joint3,0.0825,0.316,pi/2,q3,转动关节
Joint4,-0.0825,0,-pi/2,q4,转动关节
Joint5,0,0.384,pi/2,q5,转动关节
Joint6,0.088,0,pi/2,q6,转动关节
Joint7,0,0,0,q7,转动关节


## 2. 末端位姿与 MuJoCo 对照

In [3]:
model = mujoco.MjModel.from_xml_path(str(panda_xml_path()))
data = mujoco.MjData(model)
link7 = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "link7")


def rot_err(Ra, Rb):
    return float(np.linalg.norm(Ra @ Rb.T - np.eye(3)))


def mj_link7(q):
    data.qpos[:] = 0.0
    data.qpos[:7] = q
    mujoco.mj_forward(model, data)
    return data.xpos[link7].copy(), data.xmat[link7].reshape(3, 3).copy()


home = np.array(model.key("home").qpos[:7], dtype=float)
rng = np.random.default_rng(0)
cfgs = {
    "全零": np.zeros(7),
    "home": home,
    "全0.2": np.full(7, 0.2),
    "自定义A": np.array([0.1, -0.5, 0.2, -1.0, 0.3, 0.8, -0.4]),
    "自定义B": np.array([-0.3, 0.6, -0.4, -1.2, 0.5, 1.0, 0.2]),
    "随机1": rng.uniform(-1.0, 1.0, size=7),
    "随机2": rng.uniform(-1.2, 1.2, size=7),
}

models = {
    "MDH": panda_mdh,
    "DH": panda_dh,
    "PoE": panda_poe,
    "PoE→MDH": mdh_from_poe,
    "PoE→DH": dh_from_poe,
}

hdr = f"{'配置':<8}" + "".join(f"{k+'↔MJ':>12}" for k in models)
print(hdr)
print("-" * len(hdr))
for name, q in cfgs.items():
    p_mj, _ = mj_link7(q)
    row = f"{name:<8}"
    for rob in models.values():
        T = rob.fk_numpy(q)
        row += f"{np.linalg.norm(T[:3, 3] - p_mj):12.3e}"
    print(row)

print("\n姿态误差:")
print(hdr.replace("↔MJ", "rot"))
print("-" * len(hdr))
for name, q in cfgs.items():
    _, R_mj = mj_link7(q)
    row = f"{name:<8}"
    for rob in models.values():
        T = rob.fk_numpy(q)
        row += f"{rot_err(T[:3, :3], R_mj):12.3e}"
    print(row)

q = cfgs["自定义A"]
p_mj, _ = mj_link7(q)
print("\n示例 自定义A 末端位置:")
print("  MuJoCo :", p_mj)
for k, rob in models.items():
    print(f"  {k:<8}:", rob.fk_numpy(q)[:3, 3])

配置            MDH↔MJ       DH↔MJ      PoE↔MJ  PoE→MDH↔MJ   PoE→DH↔MJ
--------------------------------------------------------------------
全零         6.939e-17   6.939e-17   6.939e-17   6.939e-17   6.939e-17
home       3.511e-16   3.511e-16   1.110e-16   3.511e-16   3.511e-16
全0.2       2.307e-16   2.307e-16   4.495e-16   2.307e-16   2.307e-16
自定义A       5.303e-16   5.303e-16   7.340e-16   5.303e-16   5.303e-16
自定义B       7.468e-16   7.468e-16   6.004e-16   7.468e-16   7.468e-16
随机1        3.469e-16   3.469e-16   5.663e-16   3.469e-16   3.469e-16
随机2        1.963e-16   1.963e-16   1.777e-16   1.963e-16   1.963e-16

姿态误差:
配置            MDHrot       DHrot      PoErot  PoE→MDHrot   PoE→DHrot
--------------------------------------------------------------------
全零         1.538e-15   1.538e-15   1.538e-15   1.538e-15   1.538e-15
home       1.756e-15   1.756e-15   1.756e-15   1.756e-15   1.756e-15
全0.2       3.503e-16   3.503e-16   3.503e-16   3.503e-16   3.503e-16
自定义A       2.395e-15   2.39

## 3. 逐连杆位姿

MDH 连杆帧与 MJCF 的 `link1`～`link7` 对齐。`T_by_link` 保存相对基座的齐次变换。

In [ ]:
q_link = cfgs["自定义A"]
data.qpos[:] = 0.0
data.qpos[:7] = q_link
mujoco.mj_forward(model, data)
T_by_link = {}
for i, name in enumerate(PANDA_MDH_BODY_NAMES):
    T_link = panda_mdh.fk_numpy(q_link, link=i)
    T_by_link[name] = T_link
    bid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, name)
    pos_error = np.linalg.norm(T_link[:3, 3] - data.xpos[bid])
    rot_error = rot_err(T_link[:3, :3], data.xmat[bid].reshape(3, 3))
    print(f"{name}: position={pos_error:.3e}, rotation={rot_error:.3e}")
    print(T_link)


## 4. 几何雅可比

In [ ]:
def J_ours(robot, q):
    fun = sympy.lambdify(list(robot.q), robot.kinematics.J[-1], "numpy")
    return np.asarray(fun(*q), dtype=float)  # 6x7 = [Jp; Jo]


def J_mujoco(q):
    data.qpos[:] = 0.0
    data.qpos[:7] = q
    mujoco.mj_forward(model, data)
    jacp = np.zeros((3, model.nv))
    jacr = np.zeros((3, model.nv))
    mujoco.mj_jacBody(model, data, jacp, jacr, link7)
    # 取手臂的 7 列，按线速度、角速度排列。
    return np.vstack([jacp[:, :7], jacr[:, :7]])


robots = {"MDH": panda_mdh, "DH": panda_dh, "PoE": panda_poe}

print(f"{'配置':<8}" + "".join(f"{k+'↔MJ':>12}" for k in robots))
print("-" * 44)
for name, q in cfgs.items():
    Jm = J_mujoco(q)
    row = f"{name:<8}"
    for rob in robots.values():
        row += f"{np.linalg.norm(J_ours(rob, q) - Jm):12.3e}"
    print(row)

q = cfgs["自定义A"]
Jm = J_mujoco(q)
Jo = J_ours(panda_mdh, q)
print("\n自定义A: max|MDH-MJ| =", np.max(np.abs(Jo - Jm)))
print("MuJoCo J:\n", np.array2string(Jm, precision=4, suppress_small=True))

## 5. 解析雅可比

位置加 roll/pitch/yaw 的解析雅可比，与几何雅可比转换结果比较。

In [ ]:
kin = panda_mdh.kinematics

cfgs = {
    "全零": np.zeros(7),
    "全0.2": np.full(7, 0.2),
    "自定义A": np.array([0.1, -0.5, 0.2, -1.0, 0.3, 0.8, -0.4]),
    "自定义B": np.array([-0.3, 0.6, -0.4, -1.2, 0.5, 1.0, 0.2]),
}

print(f"{'配置':<8} {'‖Ja-Ja_c‖':>12} {'‖Δp‖':>10} {'‖Δφ‖':>10}")
print("-" * 44)
for name, q in cfgs.items():
    Ja = kin.ja_numpy(q)                    # 直接对 xa 差分
    Ja_c = kin.ja_from_geometric_numpy(q)   # 几何 → 解析（对照）
    d = Ja - Ja_c
    print(f"{name:<8} {np.linalg.norm(d):12.3e} "
          f"{np.linalg.norm(d[:3]):10.3e} {np.linalg.norm(d[3:]):10.3e}")

q = cfgs["自定义A"]
Ja = kin.ja_numpy(q)
print("\n自定义A 解析 Ja (6×7)：")
print(np.array2string(Ja, precision=4, suppress_small=True))
print("\nxa = [p; roll,pitch,yaw]，Ja 满足 ẋa = Ja q̇")